# Calculations and demos of the Fourier grid generation

In [ ]:
import os
import numpy as np

import seaborn as sns
import matplotlib.cm as cm
import matplotlib.pyplot as plt

In [ ]:
import logging

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

In [ ]:
ODIR = 'output'
FFMT = 'png'

## Fourier grid with cubic voxels

In [ ]:
def calculate_voxels(nmesh, Lbox, ref_L):
    '''
    
    Parameters
    ----------
    nmesh : int
        Number of voxels in a selected dimension.
    Lbox : list of float
        Length of the box in each dimension [Lx, Ly, Lz].
    ref_L : float or tuple of float
        Reference length for the mesh calculation. If a tuple is
        provided, it should contain two values: (ref_L1, ref_L2).
    silent : bool
        If True, suppresses output messages.
    '''
    if type(ref_L) is tuple:
        ref_L1, ref_L2 = ref_L
    else:
        ref_L1 = ref_L2 = ref_L
    nvox = np.ceil(Lbox / (ref_L1 / nmesh))
    nvox = (nvox + nvox % 2).astype(int)  # Ensure even number of voxels
    dk = ref_L2 / nvox[Lbox.index(ref_L2)]
    log.info('mesh: Nx={}, Ny={}, Nz={}; step size: {}'.format(*nvox, dk))
    return nvox, dk

In [ ]:
def cubic_voxels_full(nmesh, Lbox, fix='max'):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.

    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on either the longest or the shortest dimension
    of the cuboid and scales the other dimensions accordingly.
    
    Parameters
    ----------
    nmesh : int
        Number of voxels in a selected dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz].
    fix : str
        - If 'maxl', the longest dimension is used to determine the
        number of voxels and the length of the other dimensions is
        scaled.
        - If 'minl', the shortest dimension is used.
        - If 'minn', the number of voxels is set to the number of voxels
          along the shortest dimension of the box.
    '''
    if not (np.isscalar(Lbox) or (hasattr(Lbox, '__len__') and len(Lbox) == 3)):
        raise TypeError('Lbox must be a scalar or a container of 3 elements!')
    Lbox = np.broadcast_to(Lbox, (3,))

    if fix.lower() == 'maxl':
        ref_L = np.max(Lbox)
    elif fix.lower() == 'minl':
        ref_L = tuple((np.max(Lbox), np.min(Lbox)))
    elif fix.lower() == 'minn':
        ref_L = np.min(Lbox)
    else:
        raise ValueError("Argument ``fix`` must be 'maxl', 'minl' or 'minn'.")

    nvox, dk = calculate_voxels(nmesh, Lbox, ref_L)
    Lbox_new = [ni * dk for ni in nvox]
    log.info('Adjusted box size: Lx={}, Ly={}, Lz={}'.format(*Lbox_new))
    return nvox, dk, Lbox_new

In [ ]:
Lbox = [1000.0, 1000.0, 50.0]
periodic = [0, 0, 1]

nmesh = 128
_, _, _ = cubic_voxels_full(nmesh, Lbox, fix='maxl')

print()

nmesh = 128
_, _, _ = cubic_voxels_full(nmesh, Lbox, fix='minl')

print()

nmesh = 8
_, _, _ = cubic_voxels_full(nmesh, Lbox, fix='minn')

In [ ]:
def cubic_voxels(nmesh, Lbox):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters
    ----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.

    Returns
    -------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    '''
    if not (np.isscalar(Lbox) or (hasattr(Lbox, '__len__') and len(Lbox) == 3)):
        raise TypeError('Lbox must be a scalar or a container of 3 elements!')
    Lbox = np.broadcast_to(Lbox, (3,))
    nvox = np.ceil(Lbox / (np.min(Lbox) / nmesh)).astype(int)
    nvox = (nvox + nvox % 2).astype(int)  # Ensure even number of voxels
    dk = np.min(Lbox) / np.min(nvox)
    log.info('mesh: Nx={}, Ny={}, Nz={}; step size: {}'.format(*nvox, dk))
    return nvox, dk

In [ ]:
def fourier_grid(nvox, dk, hermitian=False):
    r'''
    Construct a 3D Fourier space grid.

    This function generates a three-dimensional array of wavevector
    components (``kvec``) and computes the corresponding magnitude
    (``kmod``) for a cubic grid with ``nmesh`` points per side within
    a box of size ``Lbox``.

    The grid is then constructed using the FFT frequencies:
    - For the first two dimensions, the full set of FFT frequencies is
      computed using ``np.fft.fftfreq``.
    - For the third dimension, if the input field is real-valued (i.e.
      if Hermitian symmetry is assumed), the reduced set of frequencies
      is computed using ``np.fft.rfftfreq``.

    Parameters
    ----------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    hermitian : bool
        If True, assume the field has Hermitian symmetry (i.e., it is
        real-valued) and use the reduced FFT along the last dimension.

    Returns
    -------
    kvec : ndarray
        A three-dimensional array of wavevector components with shape:
          - :math:`(3, {N_x}, {N_y}, {N_z}//2+1)` if ``hermitian`` is True.
          - :math:`(3, {N_x}, {N_y}, {N_z})` if ``hermitian`` is False.
        Each sub-array corresponds to the ``x``, ``y``, or ``z`` component
        of the wavevector.
    
    kmod : ndarray
        The magnitude of the wavevector at each grid point, computed as
        :math:`\|\mathbf{k}\| = \sqrt{k_x^2 + k_y^2 + k_z^2}`.
    '''
    kx = np.fft.fftfreq(nvox[0]) * 2 * np.pi / dk
    ky = np.fft.fftfreq(nvox[1]) * 2 * np.pi / dk
    if hermitian:
        kz = np.fft.rfftfreq(nvox[2]) * 2 * np.pi / dk
    else:
        kz = np.fft.fftfreq(nvox[2]) * 2 * np.pi / dk
    kvec = np.array(np.meshgrid(kx, ky, kz, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod

In [ ]:
%%timeit -n 100 -r 5
nvox, dk = cubic_voxels(nmesh=128, Lbox=1000.0)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)

In [ ]:
%%timeit -n 100 -r 5
nvox, dk = cubic_voxels(nmesh=128, Lbox=1000.0)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)

In [ ]:
%%timeit -n 100 -r 5
nvox, dk = cubic_voxels(nmesh=8, Lbox=[1000, 1000, 50])
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)

In [ ]:
%%timeit -n 100 -r 5
nvox, dk = cubic_voxels(nmesh=8, Lbox=[1000, 1000, 50])
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)

In [ ]:
nvox, dk = cubic_voxels(nmesh=128, Lbox=1000.0)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
print('kvec.sum:              ', kvec.sum())
print('kvec.shape:            ', kvec.shape)
print('kmod.shape:            ', kmod.shape)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
print('kvec.sum (Hermitian):  ', kvec.sum())
print('kvec.shape (Hermitian):', kvec.shape)
print('kmod.shape (Hermitian):', kmod.shape)

In [ ]:
nvox, dk = cubic_voxels(nmesh=8, Lbox=[1000, 1000, 50])
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
print('kvec.sum:              ', kvec.sum())
print('kvec.shape:            ', kvec.shape)
print('kmod.shape:            ', kmod.shape)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
print('kvec.sum (Hermitian):  ', kvec.sum())
print('kvec.shape (Hermitian):', kvec.shape)
print('kmod.shape (Hermitian):', kmod.shape)

### Plot wavenumbers

In [ ]:
def plot_wavenumber(ax, nmesh, Lbox):
    '''TODO'''
    kk = np.fft.fftfreq(nmesh) * 2*np.pi / Lbox * nmesh
    ks = np.fft.rfftfreq(nmesh) * 2*np.pi / Lbox * nmesh
    ax.plot(kk, label='k', color='0.2', lw=3)
    ax.plot(ks, label='k [real]', color='0.8', lw=4, ls=':')
    ax.set_xlabel(f'Grid index [0-{nmesh}]')
    ax.set_ylabel('Wavenumber $k$ [h/Mpc]')
    ax.text(0.05, 0.05, f'nmesh = {nmesh}\nLbox = {Lbox} Mpc/h', fontsize=8,
            transform=ax.transAxes, va='bottom', ha='left',
            bbox=dict(boxstyle='square,pad=0.3', ec='black', fc='none', alpha=0.5))
    ax.legend(loc='upper right', fontsize=8)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

plot_wavenumber(axes[0], nmesh=128, Lbox=1_000)
plot_wavenumber(axes[1], nmesh=256, Lbox=2_000)
plot_wavenumber(axes[2], nmesh=384, Lbox=3_000)

os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_grid_varying_N_L.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

#### Fix `nmesh`

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

nmesh = 256   # number of mesh points
plot_wavenumber(axes[0], nmesh=nmesh, Lbox=1_000)
plot_wavenumber(axes[1], nmesh=nmesh, Lbox=2_000)
plot_wavenumber(axes[2], nmesh=nmesh, Lbox=3_000)

os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_grid_varying_L.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

#### Fix `Lbox`

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

Lbox = 1_000  # box size in [Mpc/h]
plot_wavenumber(axes[0], nmesh=128, Lbox=Lbox)
plot_wavenumber(axes[1], nmesh=256, Lbox=Lbox)
plot_wavenumber(axes[2], nmesh=384, Lbox=Lbox)

os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_grid_varying_N.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

### Plot wavevectors

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
def calculate_kvec_slice(kvec, kmod, slice_idx, step=4, axis=None):
    '''TODO'''
    sl_step = slice(None, None, step)
    index = [sl_step, sl_step, sl_step]
    if axis in [0, 1, 2]:
        index[axis] = slice_idx
    else:
        raise ValueError("Axis must be specified (0, 1, or 2).")

    ki = tuple(i for i in range(3) if i != axis)
    k1_s = kvec[ki[0]][tuple(index)]
    k2_s = kvec[ki[1]][tuple(index)]
    kmod_s = kmod[tuple(index)]
    return k1_s, k2_s, kmod_s

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.5, nr*4), dpi=120)
fig.subplots_adjust(wspace=-0.7)

nmesh, Lbox = 128, [1000, 1000, 1000]
nvox, dk = cubic_voxels(nmesh, Lbox)
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)

slice_idx = kmod.shape[2]//2

# quiver along an axis at fixed kx or ky or kz
ax = axes[0]
k1_s, k2_s, kmod_s = calculate_kvec_slice(
        kvec, kmod, slice_idx=slice_idx, step=4, axis=2)
ax.quiver(k1_s, k2_s, k1_s/kmod_s, k2_s/kmod_s, kmod_s, # [X, Y], [U, V], [C]
          cmap='plasma', scale=None, width=0.003)
ax.set_title(f'Vector slice at $k_z$ index={slice_idx}',
            loc='left', fontsize=10)
dx, dy = k1_s.max() - k1_s.min(), k2_s.max() - k2_s.min()
x_min, x_max = k1_s.min()-0.05*dx, k1_s.max()+0.05*dx
y_min, y_max = k2_s.min()-0.05*dy, k2_s.max()+0.05*dy

# scalar contour of kmod
ax = axes[1]
levels = np.linspace(0.3, 0.5, 100, endpoint=True)
c = ax.contourf(k1_s, k2_s, kmod_s, levels=levels)
text = f'Scalar slice of $|\\mathbf{{k}}|$ at $k_z$ index={slice_idx}\n' \
       f'Plotting $|\\mathbf{{k}}| \\in [0.3, 0.5]$ only'
ax.set_title(text, loc='left', fontsize=10)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
cb = plt.colorbar(c, cax=cax)
cb.set_ticks(np.linspace(0.3, 0.5, 5))
cb.set_label(r'$|\mathbf{k}|$ [h/Mpc]', rotation=270, labelpad=15)

for ax in axes.flat:
    ax.set_xlabel(r'$k_x$ [h/Mpc]')
    ax.set_ylabel(r'$k_y$ [h/Mpc]')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')

plt.tight_layout()
os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_vec_scalar_slice.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
def radial_profile(kmod):
    '''TODO
    
    Tests TODO:
    1. slope of left side
        - x-y dimension: k^2
        - x-y-z dimension: k^3
    '''
    ks = kmod.ravel()
    bins = np.linspace(0, ks.max(), 100)
    hist, edges = np.histogram(ks, bins=bins)
    centers = 0.5*(edges[:-1]+edges[1:])
    return centers, hist

In [ ]:
def grid_info(ax, nmesh, Lbox):
    nvox, _ = cubic_voxels(nmesh, Lbox)
    info = f'Mesh = {nvox}\nLbox = {Lbox} Mpc/h'
    text = f'\nMin. res. = {2*np.pi/kmod.max():.2f} Mpc/h'
    ax.text(0.025, 0.875, info+text, fontsize=8,
            transform=ax.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='square,pad=0.3', ec='black', fc='none', alpha=0.5))

In [ ]:
def plot_nyquist(ax, nmesh, Lbox):
    _, dk = cubic_voxels(nmesh, Lbox)
    k_nyquist = np.min(np.pi / dk)  # Limiting minimum Nyquist frequency
    ax.axvline(x=k_nyquist, color='r', linestyle=':', label='Nyquist frequency')

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120, sharey=True)
fig.subplots_adjust(wspace=0.05)

nmesh, Lbox = 128, 1000
_, dk = cubic_voxels(nmesh, Lbox)

ax = axes[0]
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid', color='0.3')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'$|\mathbf{k}|$ [h/Mpc]')
ax.set_ylabel('Number of $k$-modes')
ax.set_title('Radial profile of non-Hermitian $k$-modes', loc='left', fontsize=10)
ax.legend(loc='upper left', fontsize=8)
grid_info(ax, nmesh, Lbox)

ax = axes[1]
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid', color='0.3')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'$|\mathbf{k}|$ [h/Mpc]')
ax.set_title('Radial profile of Hermitian $k$-modes', loc='left', fontsize=10)
ax.legend(loc='upper left', fontsize=8)
grid_info(ax, nmesh, Lbox)

os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_radial_profile_cube.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120, sharey=True)
fig.subplots_adjust(wspace=0.05)

nmesh, Lbox = 8, [1000, 1000, 50]
nvox, dk = cubic_voxels(nmesh, Lbox)

ax = axes[0]
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid', color='0.3')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'$|\mathbf{k}|$ [h/Mpc]')
ax.set_ylabel('Number of $k$-modes')
ax.set_title('Radial profile of non-Hermitian $k$-modes', loc='left', fontsize=10)
ax.legend(loc='upper left', fontsize=8)
grid_info(ax, nmesh, Lbox)

ax = axes[1]
kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid', color='0.3')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'$|\mathbf{k}|$ [h/Mpc]')
ax.set_title('Radial profile of Hermitian $k$-modes', loc='left', fontsize=10)
ax.legend(loc='upper left', fontsize=8)
grid_info(ax, nmesh, Lbox)

os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_radial_profile_cuboid.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
from skimage import measure
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
def visualize_kmod_isosurfaces(
            ax, kmod, levels=[0.1, 0.3, 0.5], xmin=0.3, xmax=0.7, title=None):
    kmod_norm = kmod / kmod.max()

    lmap = np.interp(levels, [min(levels), max(levels)], [xmin, xmax])
    # Create isosurfaces at different levels
    for level, color in zip(levels, lmap): 
        verts, faces, _, _ = measure.marching_cubes(kmod_norm, level=level)

        v = (verts[:, 0], verts[:, 1], faces, verts[:, 2])
        ax.plot_trisurf(*v, label=f'k={level:.1f}', color=cm.magma(color),
                        edgecolor='none', alpha=0.3)

    ax.set_xlabel('kx')
    ax.set_ylabel('ky')
    ax.set_zlabel('kz')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_title(title, fontsize=10)

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*6), dpi=120,
                         subplot_kw={'projection': '3d'})
for ax in axes.flat:
    ax.set_box_aspect(None, zoom=0.85)
    ax.view_init(elev=45, azim=35)

nmesh, Lbox = 128, 1000
nvox, dk = cubic_voxels(nmesh, Lbox)

kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[0], kmod=kmod, levels=levels, xmin=0.1, xmax=0.9,
                           title='Isosurfaces of non-Hermitian $k$-modes')

kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[1], kmod=kmod, levels=levels, xmin=0.1, xmax=0.9,
                           title='Isosurfaces of Hermitian $k$-modes')

fig.tight_layout()
os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_isosurfaces_cube.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*6), dpi=120,
                         subplot_kw={'projection': '3d'})
for ax in axes.flat:
    ax.set_box_aspect(None, zoom=0.85)
    ax.view_init(elev=45, azim=35)

nmesh, Lbox = 8, [1000, 1000, 50]
nvox, dk = cubic_voxels(nmesh, Lbox)

kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=False)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[0], kmod=kmod, levels=levels, xmin=0.1, xmax=0.9,
                           title='Isosurfaces of non-Hermitian $k$-modes')

kvec, kmod = fourier_grid(nvox=nvox, dk=dk, hermitian=True)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[1], kmod=kmod, levels=levels, xmin=0.1, xmax=0.9,
                           title='Isosurfaces of Hermitian $k$-modes')

fig.tight_layout()
os.makedirs(ODIR, exist_ok=True)
fig.savefig(os.path.join(ODIR, f'k_isosurfaces_cuboid.{FFMT}'),
            bbox_inches='tight', dpi=120)
plt.show()